<a href="https://colab.research.google.com/github/TejashwiniByrappa/Tejashwini-landing-page/blob/main/titanic_survival_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ============================================================
# 1. Install libraries
# ============================================================
!pip -q install transformers accelerate sentencepiece datasets kagglehub
# ============================================================
# 2. Imports and model setup
# ============================================================
import re
import random
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import accuracy_score, classification_report
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("Device:", next(model.parameters()).device)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Device: cuda:0


In [5]:
# ============================================================
# 3. Download the Titanic dataset
# ============================================================
import kagglehub

dataset_path = kagglehub.dataset_download(
    "yasserh/titanic-dataset"
)

print("Dataset path:", dataset_path)
# ============================================================
# 4. Load the CSV file
# ============================================================
import glob
import os
import pandas as pd
csv_files = glob.glob(
    os.path.join(dataset_path, "**", "*.csv"),
    recursive=True
)

print("CSV files:", csv_files)

# Select the CSV containing the Survived target column
df = None

for file in csv_files:
    candidate = pd.read_csv(file)

    if "Survived" in candidate.columns:
        df = candidate
        print("Using:", file)
        break

if df is None:
    raise FileNotFoundError(
        "No CSV containing the 'Survived' column was found."
    )

print(df.shape)
df.head()
# ============================================================
# 5. Clean the data
# ============================================================
def clean_value(value, default="unknown"):
    if pd.isna(value):
        return default
    return str(value).strip()


def passenger_to_text(row):
    sex = clean_value(row.get("Sex"))
    age = clean_value(row.get("Age"))
    passenger_class = clean_value(row.get("Pclass"))
    siblings_spouses = clean_value(row.get("SibSp"), "0")
    parents_children = clean_value(row.get("Parch"), "0")
    fare = clean_value(row.get("Fare"))
    embarked = clean_value(row.get("Embarked"))
    name = clean_value(row.get("Name"))

    return (
        f"Name: {name}\n"
        f"Sex: {sex}\n"
        f"Age: {age}\n"
        f"Passenger class: {passenger_class}\n"
        f"Siblings or spouses aboard: {siblings_spouses}\n"
        f"Parents or children aboard: {parents_children}\n"
        f"Ticket fare: {fare}\n"
        f"Embarkation port: {embarked}"
    )


df = df.dropna(subset=["Survived"]).copy()
df["Survived"] = df["Survived"].astype(int)

df[["Survived", "Sex", "Age", "Pclass"]].head()
# ============================================================
# 6. Create the classification prompt
# ============================================================
SYSTEM_PROMPT = """
You are a binary classification model.

Predict whether a Titanic passenger survived.

Output exactly one digit:
1 = survived
0 = did not survive

Do not provide explanations or additional text.
""".strip()


def create_prompt(row):
    passenger = passenger_to_text(row)

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": (
                "Classify this passenger:\n\n"
                f"{passenger}\n\n"
                "Prediction:"
            )
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
# ============================================================
# 7. Prediction function
# ============================================================
def predict_survival(row):
    prompt = create_prompt(row)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=3,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    match = re.search(r"\b([01])\b", answer)

    if match:
        return int(match.group(1))

    # Conservative fallback if the response is malformed
    return 0
# ============================================================
# 8. Test one passenger
# ============================================================
example = df.iloc[0]

print(passenger_to_text(example))
print("\nActual:", example["Survived"])
print("Predicted:", predict_survival(example))
# ============================================================
# 9. Evaluate a sample
# ============================================================
# Increase SAMPLE_SIZE after confirming that everything works.
# LLM inference over all rows can take several minutes.

SAMPLE_SIZE = min(100, len(df))

evaluation_df = df.sample(
    n=SAMPLE_SIZE,
    random_state=SEED
).copy()

predictions = []

for _, row in tqdm(
    evaluation_df.iterrows(),
    total=len(evaluation_df)
):
    predictions.append(predict_survival(row))

evaluation_df["LLM_Prediction"] = predictions

accuracy = accuracy_score(
    evaluation_df["Survived"],
    evaluation_df["LLM_Prediction"]
)

print(f"Accuracy: {accuracy:.4f}")

print(
    classification_report(
        evaluation_df["Survived"],
        evaluation_df["LLM_Prediction"],
        digits=4,
        zero_division=0
    )
)
# ============================================================
# 10. Inspect and save results
# ============================================================
display(
    evaluation_df[
        [
            "Name",
            "Sex",
            "Age",
            "Pclass",
            "Survived",
            "LLM_Prediction"
        ]
    ].head(20)
)

evaluation_df.to_csv(
    "titanic_llm_predictions.csv",
    index=False
)

print("Saved: titanic_llm_predictions.csv")
new_passenger = pd.Series({
    "Name": "Example Passenger",
    "Sex": "female",
    "Age": 28,
    "Pclass": 1,
    "SibSp": 0,
    "Parch": 1,
    "Fare": 80.0,
    "Embarked": "C"
})

prediction = predict_survival(new_passenger)

print("Survived" if prediction == 1 else "Did not survive")


Using Colab cache for faster access to the 'titanic-dataset' dataset.
Dataset path: /kaggle/input/titanic-dataset
CSV files: ['/kaggle/input/titanic-dataset/Titanic-Dataset.csv']
Using: /kaggle/input/titanic-dataset/Titanic-Dataset.csv
(891, 12)
Name: Braund, Mr. Owen Harris
Sex: male
Age: 22.0
Passenger class: 3
Siblings or spouses aboard: 1
Parents or children aboard: 0
Ticket fare: 7.25
Embarkation port: S

Actual: 0
Predicted: 0


  0%|          | 0/100 [00:00<?, ?it/s]

Accuracy: 0.6600
              precision    recall  f1-score   support

           0     0.6477    0.9500    0.7703        60
           1     0.7500    0.2250    0.3462        40

    accuracy                         0.6600       100
   macro avg     0.6989    0.5875    0.5582       100
weighted avg     0.6886    0.6600    0.6006       100



,Name,Sex,Age,Pclass,Survived,LLM_Prediction
709,"Moubarek, Master. Halim Gonios (""William George"")",male,NaN,3,1,0
439,"Kvillner, Mr. Johan Henrik Johannesson",male,31.0,2,0,0
840,"Alhomaki, Mr. Ilmari Rudolf",male,20.0,3,0,0
720,"Harper, Miss. Annie Jessie ""Nina""",female,6.0,2,1,0
39,"Nicola-Yarred, Miss. Jamila",female,14.0,3,1,0
290,"Barber, Miss. Ellen ""Nellie""",female,26.0,1,1,0
300,"Kelly, Miss. Anna Katherine ""Annie Kate""",female,NaN,3,1,0
333,"Vander Planke, Mr. Leo Edmondus",male,16.0,3,0,0
208,"Carr, Miss. Helen ""Ellen""",female,16.0,3,1,0
136,"Newsom, Miss. Helen Monypeny",female,19.0,1,1,0


Saved: titanic_llm_predictions.csv
Survived
